# Example of the workflow to see if a shot is faulty

TODO: find a better way to download the data, as of now the data are being transfered to a specific directory after having been read via the gui_qxt_loader

In [ ]:
# Import needed libraries
import os
import h5py
import numpy as np
import seaborn as sns
%matplotlib widget
import matplotlib.pyplot as plt
import torch

from dataset import AEDataModule
from model import AutoEncoder
from read_h5f import load_sxr_data
from anomaly_detection import detect_anomalies

sns.set_theme(context="notebook", style="whitegrid", palette="muted", 
              font="sans-serif", font_scale=1.2, color_codes=True, rc=None)

In [ ]:
# Load the model and the dataset
v_num = 0
model_path = f"/home/IPP-HGW/orluca/devel/classificator/W7-X_QXT/AE360/version_{v_num}/best_model_.ckpt"
model = AutoEncoder.load_from_checkpoint(model_path)
model.eval() # Set the model to evaluation mode


# # TODO: make the min_val and max_val variables of the model so not to have to load
# # the datamodule just to get those values
# data_dir = '../data'
# filename = '20251028_all_data.npz'
# data_module = AEDataModule(data_dir, filename, batch_size=-1, normalization_strategy='minmax')
# data_module.setup()

# # get from the datamodule the values for the normalization of the data
# min_data = np.array(data_module.min_val)
# max_data = np.array(data_module.max_val)

min_data = -0.2294265627861023
max_data = 8.772955894470215

print(f"min_data: {min_data}")
print(f"max_data: {max_data}")

In [ ]:
# Now get the data from the pid we want to analyze
pid = "20250401.49" # This is a pid with an anomaly
# pid = "20230305.50"
data = load_sxr_data(pid)
# find the maximum of the emission in data
max_emission = np.argmax(data[152, :]) # Here I am using diode #151
# once I have the index of the maximum, I can find the corresponding time instant
time_instant = data[0, max_emission] # Here I am using the time base which is in the first row of the data
print(f"Time instant of the maximum emission: {time_instant} ms")
profile = data[1:, max_emission]

In [ ]:
# Now pass the profile through the model to get the reconstructed profile
# but first we need to normalize the profile using the min and max values from the datamodule
profile_normalized = (profile - min_data) / (max_data - min_data)
# Now we can pass the normalized profile through the model
profile_tensor = torch.tensor(profile_normalized, dtype=torch.float32).unsqueeze(0)
reconstructed_profile = model(profile_tensor).squeeze(0).detach().numpy()
# Denormalize the reconstructed profile
dn_rec_prof = reconstructed_profile * (max_data - min_data) + min_data
# Now we can plot the original and reconstructed profiles
plt.figure(figsize=(10, 5))
plt.plot(profile, label='Original Profile')
plt.plot(dn_rec_prof, label='Reconstructed Profile')
plt.legend()
plt.title(f"Original vs Reconstructed Profile at time {time_instant} ms")
plt.suptitle(f"PID: {pid}")
plt.xlabel("Diode Index")
plt.ylabel("Signal [V]")
plt.show()
# plt.close("all")


In [ ]:
# Now compute the correlation between the original and reconstructed profiles
correlation = np.corrcoef(profile, dn_rec_prof)[0, 1]
print(f"Correlation between original and reconstructed profile: {correlation:.4f}")

# if the correlation is lower than 0.9, then we flag this profile as an anomaly
if correlation < 0.9:
    print("Anomaly detected in the profile!")
else:
    print("Profile is normal.")


In [ ]:
# Now i would like to get a global figure of the correlation over the entire time series
data_normalized = (data[1:, :] - min_data) / (max_data - min_data)
data_tensor = torch.tensor(data_normalized.T, dtype=torch.float32)
reconstructed_data = model(data_tensor).detach().numpy().T
dn_rec_data = reconstructed_data * (max_data - min_data) + min_data
correlations = []
for i in range(data.shape[1]):
    corr = np.corrcoef(data[1:, i], dn_rec_data[:, i])[0, 1]
    correlations.append(corr)

In [ ]:
# Now we can plot the correlation over time
plt.figure(figsize=(10, 5))
plt.plot(data[0, :], correlations, label='Correlation')
plt.xlabel("Time [ms]")
plt.ylabel("Correlation", color='tab:blue')
plt.tick_params(axis='y', labelcolor='tab:blue')
plt.title(f"Correlation between original and reconstructed profiles over time")
plt.suptitle(f"PID: {pid}")

ax2 = plt.gca().twinx()
ax2.plot(data[0, :], data[152, :], color='tab:orange', label='Diode #151')
ax2.set_ylabel("Signal [V]", color='tab:orange')
ax2.tick_params(axis='y', labelcolor='tab:orange')

plt.legend()
plt.tight_layout()
plt.show()
# plt.close("all")

In [ ]:
# # Now, knowing that if there is an anomaly it is present on a diode over the entire time series,
# # I would like to understand which diodes are anomalous. 
# # To do this, I can compute the correlation between the original and reconstructed profiles for each diode over the entire time series.
# # Then I can plot the correlation for each diode to see if there are any diodes that have a low correlation, which would indicate an anomaly.
# print(data.shape)
# print(dn_rec_data.shape)
# # correlations = np.array(correlations)
# print(correlations.shape)
# print(correlations[0])
# # print(f"Correlations for all diodes: {correlations}")

gain_ratio = 5 / 2


In [ ]:
# # I know (maybe) that for pid = 20250401.49 the gain passes from 5 to 2, so the "good" measurements should be higher than 
# # the "bad" ones. Now if i find the peaks in the profile at a certain time instant, I can try and multiply the non-peak values
# # by the ratio between the old and new gain to see if the correlation improves.
# from scipy.signal import find_peaks


# # locate every local maximum
# all_peaks, _ = find_peaks(profile)

# # compute the slope of the signal
# dp = np.diff(profile)

# # choose a threshold for a “spike” (steep rise); adjust the multiplier as needed
# th = np.mean(np.abs(dp)) + 2*np.std(dp)

# # keep only peaks preceded by a very steep rise
# spike_peaks = [p for p in all_peaks
#                if p > 0 and abs(profile[p] - profile[p-1]) > th]

# peaks = np.array(spike_peaks)

# # now plot the profile and the peaks
# plt.figure(figsize=(10, 5))
# plt.plot(profile, marker='s', label='Original Profile')
# plt.plot(peaks, profile[peaks], "x", label='Peaks')
# plt.legend()
# plt.title(f"Original Profile with Peaks at time {time_instant} ms")
# plt.suptitle(f"PID: {pid}")
# plt.xlabel("Diode Index")
# plt.ylabel("Signal [V]")
# plt.show()
# # plt.close("all")

In [ ]:
# # Try an iterative version of this same process
# # I know (maybe) that for pid = 20250401.49 the gain passes from 5 to 2, so the "good" measurements should be higher than 
# # the "bad" ones. Now if i find the peaks in the profile at a certain time instant, I can try and multiply the non-peak values
# # by the ratio between the old and new gain to see if the correlation improves.
# from scipy.signal import find_peaks

# # Iteratively find peaks by excluding previously found ones
# peaks = []
# remaining_profile = profile.copy()
# while True:
#     # locate every local maximum in the remaining profile
#     all_peaks, _ = find_peaks(remaining_profile)
    
#     if len(all_peaks) == 0:
#         break
    
#     # compute the slope of the remaining signal
#     dp = np.diff(remaining_profile)
    
#     # choose a threshold for a “spike” (steep rise); adjust the multiplier as needed
#     th = np.mean(np.abs(dp)) + 1*np.std(dp)
    
#     # keep only peaks preceded by a very steep rise
#     spike_peaks = [p for p in all_peaks
#                    if p > 0 and abs(remaining_profile[p] - remaining_profile[p-1]) > th]
    
#     if len(spike_peaks) == 0:
#         break
    
#     # append the found spike peaks to the overall peaks list
#     peaks.extend(spike_peaks)
    
#     # exclude the found peaks from the remaining profile by setting them to a low value or interpolating
#     # for simplicity, set them to the minimum value or use interpolation
#     for p in spike_peaks:
#         remaining_profile[p] = np.min(remaining_profile)
    
#     # to avoid infinite loop, limit iterations or check if no new peaks
#     if len(peaks) > 100:  # arbitrary limit
#         break

# peaks = np.array(peaks)

# # now plot the profile and the peaks
# plt.figure(figsize=(10, 5))
# plt.plot(profile, marker='s', label='Original Profile')
# plt.plot(peaks, profile[peaks], "x", label='Peaks')
# plt.legend()
# plt.title(f"Original Profile with Peaks at time {time_instant} ms")
# plt.suptitle(f"PID: {pid}")
# plt.xlabel("Diode Index")
# plt.ylabel("Signal [V]")
# plt.show()
# # plt.close("all")

In [ ]:
plt.close("all")

In [ ]:
# # Now try and replot the profiles multiplying the non-peak values by the ratio between the old and new gain to see if the correlation improves.
# gain_ratio = 5 / 2
# adjusted_profile = profile.copy()
# adjusted_profile[~np.isin(np.arange(len(profile)), peaks)] *= gain_ratio
# # Now we can plot the original and adjusted profiles
# plt.figure(figsize=(10, 5))
# plt.plot(profile, label='Original Profile')
# plt.plot(adjusted_profile, label='Adjusted Profile')
# plt.scatter(peaks, profile[peaks], marker='x', color='red', label='Peaks')
# plt.legend()
# plt.title(f"Original vs Adjusted Profile at time {time_instant} ms")
# plt.suptitle(f"PID: {pid}")
# plt.xlabel("Diode Index")
# plt.ylabel("Signal [V]")
# plt.show()

In [ ]:
# # Now try and do the same using not the peaks but the detect_anomalies function that
# # I have implemented in the anomaly_detection.py file. This function should return 
# # the indices of the diodes that are anomalous, so I can use those indices to adjust 
# # the profile and see if the correlation improves.

# anomaly_mask, z_scores, roll_mean, roll_std = detect_anomalies(profile,
#                                                  window=18, z_threshold=2)

# print(f"Anomalous diodes: {np.where(anomaly_mask)[0]}")

# # plot the original profile and detected anomalies
# plt.figure(figsize=(10, 5))
# plt.plot(profile, label='Original Profile')
# plt.plot(np.where(anomaly_mask)[0], profile[anomaly_mask], "x", label='Anomalies')
# plt.legend()
# plt.title(f"Original Profile with Detected Anomalies at time {time_instant} ms")
# plt.suptitle(f"PID: {pid}")
# plt.xlabel("Diode Index")
# plt.ylabel("Signal [V]")
# plt.show()

In [ ]:
# # Now let's try and get the anomalous diodes by applying a spline fit done through smppthing_splines of scipy. 
# # I  would like to use the spline fit with the correlation check to see if I can get a better reconstruction of the profile and therefore a higher correlation.

# profile_spline = profile.copy()
# from scipy.interpolate import make_smoothing_spline

# x = np.arange(len(profile))

# # Create the smoothing spline
# spline = make_smoothing_spline(x, profile)

# # Evaluate the spline at the original points
# profile_spline = spline(x)

# # Now we can plot the original and spline-fitted profiles
# plt.figure(figsize=(10, 5))
# plt.plot(profile, label='Original Profile')
# plt.plot(profile_spline, label='Spline Fitted Profile')
# plt.legend()
# plt.title(f"Original vs Spline Fitted Profile at time {time_instant} ms")
# plt.suptitle(f"PID: {pid}")
# plt.xlabel("Diode Index")
# plt.ylabel("Signal [V]")
# plt.show()



In [ ]:
# Iteratively fit a spline, removing outliers that lie above the fit until convergence
from scipy.interpolate import make_smoothing_spline

x = np.arange(len(profile))
mask = np.ones(len(profile), dtype=bool)  # keep‑all initially
best_spline = None
best_correlation = -1

for iteration in range(20):                 # guard against infinite loops
    # fit spline on the points that are still in the mask
    spline = make_smoothing_spline(x[mask], profile[mask])
    profile_spline_fit = spline(x)

    # signed residuals: positive when the data point is above the spline
    residuals = profile - profile_spline_fit

    # threshold computed only on the upward deviations
    pos = residuals > 0
    if np.any(pos):
        th = np.mean(residuals[pos]) - .3 * np.std(residuals[pos])
    else:
        th = 0.0

    # remove only those points that are above the spline and exceed the threshold
    new_mask = ~((residuals > th) & (profile > profile_spline_fit))

    # convergence check
    if np.array_equal(mask, new_mask):
        best_spline = spline
        break

    mask = new_mask

    # keep track of the best correlation in case we want to stop earlier
    profile_normalized_temp = (profile_spline_fit - min_data) / (max_data - min_data)
    profile_tensor_temp = torch.tensor(profile_normalized_temp,
                                       dtype=torch.float32).unsqueeze(0)
    reconstructed_temp = model(profile_tensor_temp).squeeze(0).detach().numpy()
    dn_rec_temp = reconstructed_temp * (max_data - min_data) + min_data

    if mask.sum() >= 2:
        corr = np.corrcoef(profile[mask], dn_rec_temp[mask])[0, 1]
        if corr > best_correlation:
            best_correlation = corr
            best_spline = spline

# plot the results as before
profile_spline_final = best_spline(x)
outlier_mask = ~mask

plt.figure(figsize=(10, 5))
plt.plot(x, profile, label='Original Profile', marker='o', markersize=4)
plt.plot(x, profile_spline_final, label='Iterative Spline Fit', linewidth=2)
plt.scatter(x[outlier_mask], profile[outlier_mask], color='red', marker='x',
            s=100, label='Outliers', zorder=5)
plt.legend()
plt.title(f"Iterative Spline Fit at time {time_instant} ms")
plt.suptitle(f"PID: {pid}")
plt.xlabel("Diode Index")
plt.ylabel("Signal [V]")
plt.show()

print(f"Points removed: {np.sum(~mask)}")
print(f"Best correlation: {best_correlation:.4f}")


In [ ]:
# Now multiply the non-outlier points by the gain ratio and see if the correlation improves
adjusted_profile_spline = profile.copy()
adjusted_profile_spline[mask] *= gain_ratio

# Now we can plot the original and adjusted profiles
plt.figure(figsize=(16, 8))
plt.plot(profile, label='Original Profile')
plt.plot(adjusted_profile_spline, marker='+', markersize=8, label='Adjusted Profile', linewidth=2)
# plt.scatter(x[mask], profile[mask], color='green', marker='o', s=100, label='Kept Points', zorder=5)
plt.scatter(x[~mask], profile[~mask], color='red', marker='x', s=100, label='Outliers', zorder=5)
plt.legend()
plt.title(f"Original vs Adjusted Profile at time {time_instant} ms")
plt.suptitle(f"PID: {pid}")
plt.xlabel("Diode Index")
plt.ylabel("Signal [V]")
plt.show()

In [ ]:
# Here plot just the adjusted profile
plt.figure(figsize=(10, 5))
plt.plot(adjusted_profile_spline, marker='o', markersize=5, label='Adjusted Profile ', linewidth=2)
plt.legend()
plt.title(f"Adjusted Profile at time {time_instant} ms")
plt.suptitle(f"PID: {pid}")
plt.xlabel("Diode Index")
plt.ylabel("Signal [V]")
plt.show()

In [ ]:
# Now what happens if i pass this adjusted profile through the model to get the reconstructed profile and then compute the correlation?
adjusted_profile_normalized = (adjusted_profile_spline - min_data) / (max_data - min_data)
adjusted_profile_tensor = torch.tensor(adjusted_profile_normalized, dtype=torch.float32).unsqueeze(0)
reconstructed_adjusted = model(adjusted_profile_tensor).squeeze(0).detach().numpy()
dn_rec_adjusted = reconstructed_adjusted * (max_data - min_data) + min_data
correlation_adjusted = np.corrcoef(profile, dn_rec_adjusted)[0, 1]
print(f"Correlation between original and adjusted reconstructed profile: {correlation_adjusted:.4f}")

# plot also the original data and the adjusted reconstructed profile
plt.figure(figsize=(14, 8))
plt.plot(profile, label='Original Profile')
plt.plot(adjusted_profile_spline, label='Iterative Spline Fit', linewidth=2)
plt.plot(dn_rec_adjusted, label='Adjusted Reconstructed Profile')
plt.legend()
plt.title(f"Original vs Adjusted Reconstructed Profile at time {time_instant} ms")
plt.suptitle(f"PID: {pid}")
plt.xlabel("Diode Index")
plt.ylabel("Signal [V]")
plt.show()